# 03 Analysis

This notebook summarizes the main analytical results from the local DuckDB warehouse.

It covers team-level winning patterns, recent team ranking, a Thunder profile, historical player benchmarks, and recent active player tiers.

## Method Summary

- Historical benchmark pool: player-seasons with at least `750` minutes and `30` games.
- Role assignment: inferred `Creator`, `Wing`, and `Big` roles built from box-score production style.
- Benchmark cutoffs: `50th`, `75th`, and `95th` percentile thresholds within each role for `Points / 36`, `Assists / 36`, `3P Made / 36`, `Steals / 36`, `Rebounds / 36`, and `Blocks / 36`.
- Recent active-player pool: players who appeared in calendar years `2024` or `2025`, using each player's latest qualified season from the last three NBA seasons.
- Current-player qualification: latest qualified season must have at least `1000` minutes and `30` games.
- Overall tier logic: role-defining stats drive the final `50 / 75 / 95` bucket.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
import pandas as pd
from IPython.display import Image, display

from scripts.generate_visuals import (
    style_matplotlib,
    ensure_output_dir,
    get_connection,
    fetch_win_loss_summary,
    fetch_top_teams,
    fetch_thunder_profile,
    fetch_historical_role_benchmarks,
    fetch_current_player_tiers,
    plot_winning_drivers,
    plot_thunder_profile,
    plot_historical_role_benchmarks,
    plot_current_player_tiers,
)


In [ ]:
style_matplotlib()
ensure_output_dir()
con = get_connection()

## Team-Level Winning Profile

In [ ]:
win_loss = fetch_win_loss_summary(con)
win_loss

In [ ]:
winning_drivers_path = plot_winning_drivers(win_loss)
display(Image(filename=str(winning_drivers_path)))

Winning teams tend to record more assists, made shots, 3-point production, and defensive rebounds, while also committing fewer turnovers and fouls.

## Recent Team Ranking And Thunder Profile

In [ ]:
top_teams = fetch_top_teams(con)
thunder_profile = fetch_thunder_profile(con)

top_teams

In [ ]:
thunder_profile_path = plot_thunder_profile(top_teams, win_loss, thunder_profile)
display(Image(filename=str(thunder_profile_path)))

Thunder has the highest win rate since 2024 among the teams shown here. The comparison below places Thunder's winning profile next to the historical win and loss averages from the first chart.

## Historical Role Benchmarks

In [ ]:
historical_role_frame, historical_role_benchmarks = fetch_historical_role_benchmarks(con)
historical_role_benchmarks.head(9)

In [ ]:
historical_benchmark_path = plot_historical_role_benchmarks(historical_role_benchmarks)
display(Image(filename=str(historical_benchmark_path)))

This chart shows the `50th`, `75th`, and `95th` percentile benchmarks for key player stats within each role. The benchmark pool is built from qualified historical player-seasons rather than raw source position flags.

## Recent Active Player Tiers And Ranked Leaders

This section applies the historical role benchmarks to recent active players and summarizes the current pool across Low, 50, 75, and 95 overall tiers. The table also lists the top-ranked players within each role.

In [ ]:
historical_role_frame, active_players, active_window_label = fetch_current_player_tiers(con)
(
    active_players
    .assign(
        role_order=pd.Categorical(
            active_players["role"],
            categories=["Creator", "Wing", "Big"],
            ordered=True,
        )
    )
    .sort_values(["role_order", "current_rank_within_role"])
    .groupby("role", group_keys=False)
    .head(5)[["season_label", "role", "current_rank_within_role", "player", "overall_tier", "historical_percentile_within_role"]]
    .rename(
        columns={
            "season_label": "Season",
            "role": "Role",
            "current_rank_within_role": "Current Rank In Role",
            "player": "Player",
            "overall_tier": "Tier",
            "historical_percentile_within_role": "Historical Percentile In Role",
        }
    )
)

In [ ]:
active_tiers_path = plot_current_player_tiers(active_players, active_window_label)
display(Image(filename=str(active_tiers_path)))

The stacked bars show the size of each tier within the recent active-player pool, and the ranked table above shows the leading players within each role under the same benchmark rules.

## Limitations

This is a box-score benchmark model rather than a possession-level impact model. The role-based player-season method is more stable than the raw source-position approach, but it is still a simplified comparison model.